# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [3]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
openai_api_key = os.getenv('GOOGLE_API_KEY')
openai = OpenAI(api_key=openai_api_key, base_url= "https://generativelanguage.googleapis.com/v1beta/openai/")

In [5]:
# Some lists!

checklist = []
completed = []

In [6]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [7]:
get_checklist_report()

''

In [8]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [10]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [11]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [12]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [13]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [15]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [17]:
def loop(messages):
    response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [18]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [19]:
checklist, completed = [], []
loop(messages)

Checklist #1: Estimate or look up the distance between Boston and New York
Checklist #2: Calculate the position of the Boston train at 3:00 pm
Checklist #3: Calculate the remaining distance between the two trains at 3:00 pm
Checklist #4: Calculate the closing speed of the two trains moving toward each other
Checklist #5: Calculate the time required for the trains to meet after 3:00 pm
Checklist #6: Determine the exact time when they meet

The driving distance between Boston, MA and New York, NY is approximately 215 miles, but the direct 
railroad/geographical distance is typically estimated around 210 to 215 miles in standard word problems. Let's use 
215 miles as a standard realistic estimate (or sometimes textbook problems assume 210 or 220 miles; let's check 
both or use 215 miles). Actually, let's use 215 miles.

Checklist #1: Estimate or look up the distance between Boston and New York
Checklist #2: Calculate the position of the Boston train at 3:00 pm
Checklist #3: Calculate the remaining distance between the two trains at 3:00 pm
Checklist #4: Calculate the closing speed of the two trains moving toward each other
Checklist #5: Calculate the time required for the trains to meet after 3:00 pm
Checklist #6: Determine the exact time when they meet

The Boston train leaves at 2:00 pm at 60 mph. By 3:00 pm, it has traveled for 1 hour, covering 60 miles * 1 hour = 
60 miles.

Checklist #1: Estimate or look up the distance between Boston and New York
Checklist #2: Calculate the position of the Boston train at 3:00 pm
Checklist #3: Calculate the remaining distance between the two trains at 3:00 pm
Checklist #4: Calculate the closing speed of the two trains moving toward each other
Checklist #5: Calculate the time required for the trains to meet after 3:00 pm
Checklist #6: Determine the exact time when they meet

With a total distance of 215 miles and the Boston train having covered 60 miles by 3:00 pm, the remaining distance 
between the two trains at 3:00 pm is 215 - 60 = 155 miles.

... (4 outputs left out)

Checklist #1: Estimate or look up the distance between Boston and New York
Checklist #2: Calculate the position of the Boston train at 3:00 pm
Checklist #3: Calculate the remaining distance between the two trains at 3:00 pm
Checklist #4: Calculate the closing speed of the two trains moving toward each other
Checklist #5: Calculate the time required for the trains to meet after 3:00 pm
Checklist #6: Determine the exact time when they meet

Adding 1 hour and 6.43 minutes to 3:00 pm gives approximately 4:06 pm (or precisely at 4:06 pm and 26 seconds).

Checklist #1: Estimate or look up the distance between Boston and New York
Checklist #2: Calculate the position of the Boston train at 3:00 pm
Checklist #3: Calculate the remaining distance between the two trains at 3:00 pm
Checklist #4: Calculate the closing speed of the two trains moving toward each other
Checklist #5: Calculate the time required for the trains to meet after 3:00 pm
Checklist #6: Determine the exact time when they meet

Solution:

1. Distance Estimate: The geographical distance between Boston and New York is approximately 215 miles.
2. Boston Train's Position at 3:00 pm: Having traveled for 1 hour at 60 mph, the Boston train has covered 60 miles.
3. Remaining Distance: At 3:00 pm, the remaining distance between the two trains is $215 - 60 =$ 155 miles.
4. Closing Speed: With the Boston train traveling at 60 mph and the New York train traveling at 80 mph toward each 
other, their combined speed is $60 + 80 =$ 140 mph.
5. Time to Meet: The time needed to cover the remaining distance after 3:00 pm is $\frac{155 \text{ miles}}{140 
\text{ mph}} = \frac{31}{28}$ hours $\approx 1.107$ hours (or 1 hour, 6 minutes, and 26 seconds).
6. Meeting Time: Adding this duration to 3:00 pm, the two trains meet at approximately 4:06 pm (precisely at 
4:06:26 pm).

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>